# Notebook 2: PyTorch Dataset & DataLoader for VRDFormer

**Goal:** Understand how the VRDBase PyTorch Dataset loads video frames and annotations, and how the DataLoader batches them into model-ready tensors.

We'll trace through the data pipeline for both Stage 1 (pair detection) and Stage 2 (relation classification), showing exact tensor shapes at each step.

## 1. Setup & Configuration

In [ ]:
import os
import sys
import json
import torch
import numpy as np
from pathlib import Path
from argparse import Namespace

# Add repo root to path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

print(f'Root: {ROOT_DIR}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# Build a minimal args namespace that mimics the config JSON
# ADJUST paths to match your machine

def build_args(dataset='vidvrd', stage=1):
    args = Namespace()
    args.dataset = dataset
    args.stage = stage
    args.device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Paths - ADJUST THESE for your machine
    if dataset == 'vidvrd':
        args.vidvrd_path = os.environ.get('VIDVRD_PATH', '/home/zhengsipeng/data/vidvrd')
    else:
        args.vidor_path = os.environ.get('VIDOR_PATH', '/home/zhengsipeng/data/vidor')
    
    args.coco_path = ''
    
    # Model params (needed for dataset construction)
    args.num_queries = 100
    args.num_verb_classes = 132 if dataset == 'vidvrd' else 50
    args.num_obj_classes = 35 if dataset == 'vidvrd' else 80
    
    # Data params
    args.max_duration = 24
    args.seq_len = 8 if stage == 2 else 2
    args.resolution = 'large'
    args.num_workers = 0
    args.batch_size = 2 if stage == 1 else 1
    args.distributed = False
    args.debug = True  # skip raw annotation loading
    
    # Tracking params (stage 1 only)
    args.tracking = True
    args.track_prev_frame_range = 8
    args.track_prev_frame_rnd_augs = 0.01
    args.track_prev_prev_frame = False
    args.track_backprop_prev_frame = False
    
    # Transform params
    args.cautious = True   # skip RandomHorizontalFlip (buggy)
    args.by_ratio = False
    args.overflow_boxes = False
    args.deformable = False
    args.num_feature_levels = 1
    args.multi_frame_attention = False
    args.multi_frame_encoding = False
    args.position_embedding = 'sine'
    
    return args

# Create stage 1 args
args_s1 = build_args('vidvrd', stage=1)
print('Stage 1 config:')
for k, v in sorted(vars(args_s1).items()):
    if 'path' in k or k in ['dataset', 'stage', 'seq_len', 'batch_size', 'num_queries']:
        print(f'  {k}: {v}')

## 2. Build the Dataset

The dataset is constructed via `datasets/__init__.py:build_dataset()`, which dispatches to VidVRD or VidOR.

In [ ]:
from datasets import build_dataset
from datasets.dataset import VRDBase

# Build Stage 1 training dataset
print('Building Stage 1 dataset...')
dataset_s1 = build_dataset('train', args_s1)
print(f'\nDataset type: {type(dataset_s1).__name__}')
print(f'Dataset length: {len(dataset_s1)} samples')
print(f'Stage: {dataset_s1.stage}')
print(f'Seq length: {dataset_s1.seq_len}')

In [ ]:
# Inspect dataset internals
print('=== DATASET INTERNALS ===')
print(f'DB name: {dataset_s1.dbname}')
print(f'Image set: {dataset_s1.image_set}')
print(f'Data dir: {dataset_s1.data_dir}')
print(f'Anno file: {dataset_s1.anno_file}')
print(f'Num video IDs: {len(dataset_s1.video_ids)}')
print(f'\nFirst 5 video IDs: {dataset_s1.video_ids[:5]}')

# Training mode specifics
if hasattr(dataset_s1, 'train_begin_fids'):
    print(f'\nTraining clips: {len(dataset_s1.train_begin_fids)}')
    print(f'Example begin_fids: {dataset_s1.train_begin_fids[:5]}')
    print(f'Example durations: {dataset_s1.max_durations[:5]}')

## 3. Trace `__getitem__` for Stage 1

Stage 1 returns a **frame pair**: the current frame plus a previous frame packed into `target['prev_image']` and `target['prev_target']`. This is the key mechanism that enables tracking.

In [ ]:
# Get one sample from Stage 1
samples, target = dataset_s1[0]

print('=== STAGE 1 SAMPLE ===')
print(f'\n--- samples (list of frames) ---')
print(f'Type: {type(samples)}')
if isinstance(samples, (list, tuple)):
    for i, s in enumerate(samples):
        print(f'  Frame {i}: shape={tuple(s.shape)}, dtype={s.dtype}, range=[{s.min():.3f}, {s.max():.3f}]')
else:
    print(f'  Shape: {tuple(samples.shape)}, dtype={samples.dtype}')

print(f'\n--- target (single frame annotation) ---')
print(f'Type: {type(target).__name__}')
for key, value in target.items():
    if isinstance(value, torch.Tensor):
        print(f'  {key}: shape={tuple(value.shape)}, dtype={value.dtype}')
    elif isinstance(value, (list, tuple)):
        print(f'  {key}: type=list, len={len(value)}')
    else:
        print(f'  {key}: {value}')

In [ ]:
# Inspect the prev_frame data (crucial for tracking)
print('=== PREVIOUS FRAME DATA (enables tracking) ===')
prev_image = target.get('prev_image')
prev_target = target.get('prev_target')

if prev_image is not None:
    print(f'prev_image shape: {tuple(prev_image.shape)}')
    print(f'prev_image range: [{prev_image.min():.3f}, {prev_image.max():.3f}]')

if prev_target is not None:
    print(f'\nprev_target keys:')
    for key, value in prev_target.items():
        if isinstance(value, torch.Tensor):
            print(f'  {key}: shape={tuple(value.shape)}, dtype={value.dtype}')
        elif isinstance(value, (list, tuple)):
            print(f'  {key}: type=list, len={len(value)}')
        else:
            print(f'  {key}: {value}')

# Show tracking ID example
if 'so_track_ids' in target:
    print(f'\n--- Tracking ID examples ---')
    print(f'so_track_ids (first 5): {target["so_track_ids"][:5]}')
    print(f'sub_track_ids (first 5): {target["sub_track_ids"][:5]}')
    print(f'obj_track_ids (first 5): {target["obj_track_ids"][:5]}')
    if prev_target is not None and 'so_track_ids' in prev_target:
        print(f'\nprev so_track_ids (first 5): {prev_target["so_track_ids"][:5]}')
        # Check if any track IDs persist across frames
        curr_tids = set((s.item(), o.item()) for s, o in target['so_track_ids'])
        prev_tids = set((s.item(), o.item()) for s, o in prev_target['so_track_ids'])
        persistent = curr_tids & prev_tids
        print(f'Persistent track IDs: {len(persistent)} out of {len(curr_tids)}')

### Stage 1 Target Dictionary Structure

| Key | Shape | Description |
|-----|-------|-------------|
| `sub_boxes` | `(N, 4)` | Subject bounding boxes in **cxcywh** format, normalized to [0,1] |
| `obj_boxes` | `(N, 4)` | Object bounding boxes in cxcywh, normalized to [0,1] |
| `sub_labels` | `(N,)` | Subject object class index (0-34 for VidVRD, 0-79 for VidOR) |
| `obj_labels` | `(N,)` | Object class index |
| `verb_labels` | `(N, K)` | Multi-hot encoded verb labels (K=132 for VidVRD, K=50 for VidOR) |
| `sub_area` | `(N,)` | Area of subject boxes |
| `obj_area` | `(N,)` | Area of object boxes |
| `so_track_ids` | `(N, 2)` | `(sub_tid, obj_tid)` pairs linking instances across frames |
| `sub_track_ids` | `(N,)` | Subject track IDs |
| `obj_track_ids` | `(N,)` | Object track IDs |
| `orig_size` | `(2,)` | Original image dimensions `(H, W)` |
| `size` | `(2,)` | Current tensor dimensions after transforms |
| `num_inst` | int | Total number of relation instances in this frame |
| `prev_image` | `(3, H, W)` | Previous frame tensor (if tracking) |
| `prev_target` | dict | Previous frame annotations (same structure) |

**N** is the number of relation instances in the current frame (capped at `num_queries=100`).

## 4. Trace `__getitem__` for Stage 2

Stage 2 returns a **full clip** of `seq_len=8` frames. Each frame has its own target dict.

In [ ]:
# Build Stage 2 dataset
args_s2 = build_args('vidvrd', stage=2)
print('Building Stage 2 dataset...')
dataset_s2 = build_dataset('train', args_s2)
print(f'Dataset length: {len(dataset_s2)} clips')
print(f'Stage: {dataset_s2.stage}')
print(f'Seq length: {dataset_s2.seq_len}')

In [ ]:
# Get one sample from Stage 2
samples_s2, targets_s2 = dataset_s2[0]

print('=== STAGE 2 SAMPLE ===')
print(f'\n--- samples (video clip) ---')
if isinstance(samples_s2, torch.Tensor):
    print(f'Shape: {tuple(samples_s2.shape)}')
    print(f'  dim 0: seq_len ({dataset_s2.seq_len} frames)')
    print(f'  dim 1: channels (3 = RGB)')
    print(f'  dim 2-3: spatial (H, W)')
    print(f'  dtype={samples_s2.dtype}, range=[{samples_s2.min():.3f}, {samples_s2.max():.3f}]')
else:
    print(f'Type: {type(samples_s2)}')

print(f'\n--- targets (list of per-frame annotations) ---')
print(f'Type: {type(targets_s2)}')
print(f'Length: {len(targets_s2)} (one target per frame)')

# Show first frame's target
print(f'\nFrame 0 target:')
t0 = targets_s2[0]
for key, value in t0.items():
    if isinstance(value, torch.Tensor):
        print(f'  {key}: shape={tuple(value.shape)}, dtype={value.dtype}')
    elif isinstance(value, (list, tuple)):
        print(f'  {key}: type=list, len={len(value)}')
    else:
        print(f'  {key}: {value}')

In [ ]:
# Check that track IDs are consistent across frames
print('=== Cross-frame Track ID Consistency ===')
for fid in range(min(3, len(targets_s2))):
    t = targets_s2[fid]
    if 'so_track_ids' in t and len(t['so_track_ids']) > 0:
        print(f'Frame {fid}: {len(t["so_track_ids"])} relations')
        print(f'  Track IDs: {t["so_track_ids"][:3].tolist()}')
        print(f'  Sub labels: {t["sub_labels"][:3].tolist()}')
        print(f'  Obj labels: {t["obj_labels"][:3].tolist()}')

### Stage 2 Target Details

Stage 2 targets have **additional keys** beyond Stage 1:

| Extra Key | Shape | Description |
|-----------|-------|-------------|
| `unscaled_sub_boxes` | `(N, 4)` | Subject boxes in **xyxy pixel coordinates** (for ROI Align) |
| `unscaled_obj_boxes` | `(N, 4)` | Object boxes in xyxy pixel coordinates |
| `video_id` | str | Video identifier |
| `frame_id` | int | Absolute frame number in video |
| `inst_ids` | `(N,)` | Relation instance IDs (for tracking across the clip) |

The `unscaled_*` boxes are critical: they're used to extract ROI features from the encoder feature map for query initialization.

## 5. Build the DataLoader

The DataLoader uses `utils.collate_fn` to batch samples via `NestedTensor.from_tensor_list`.

In [ ]:
from torch.utils.data import DataLoader
import util.misc as utils

# Stage 1 DataLoader
loader_s1 = DataLoader(
    dataset_s1,
    batch_size=2,
    shuffle=True,
    collate_fn=utils.collate_fn,
    num_workers=0
)

print(f'Stage 1 DataLoader: {len(loader_s1)} batches')

In [ ]:
# Get a Stage 1 batch and examine shapes
samples_batch, targets_batch = next(iter(loader_s1))

print('=== STAGE 1 BATCH ===')
print(f'\nsamples type: {type(samples_batch).__name__}')

if isinstance(samples_batch, utils.NestedTensor):
    print(f'  .tensors shape: {tuple(samples_batch.tensors.shape)}')
    print(f'    dim 0: batch_size (B)')
    print(f'    dim 1: channels (3)')
    print(f'    dim 2-3: spatial (H, W)')
    print(f'  .mask shape: {tuple(samples_batch.mask.shape)}')
    print(f'  .mask dtype: {samples_batch.mask.dtype}')
    print(f'  .mask unique values: {samples_batch.mask.unique().tolist()} (True = padding)')
    
print(f'\ntargets type: {type(targets_batch)}')
print(f'targets length: {len(targets_batch)} (one dict per batch element)')
print(f'\nBatch item 0 target keys:')
for key, value in targets_batch[0].items():
    if isinstance(value, torch.Tensor):
        print(f'  {key}: shape={tuple(value.shape)}, dtype={value.dtype}, device={value.device}')

In [ ]:
# Stage 2 DataLoader (batch_size must be 1)
loader_s2 = DataLoader(
    dataset_s2,
    batch_size=1,
    shuffle=True,
    collate_fn=utils.collate_fn,
    num_workers=0
)

print(f'Stage 2 DataLoader: {len(loader_s2)} batches')

In [ ]:
# Get a Stage 2 batch
samples_s2_batch, targets_s2_batch = next(iter(loader_s2))

print('=== STAGE 2 BATCH ===')

if isinstance(samples_s2_batch, utils.NestedTensor):
    print(f'  .tensors shape: {tuple(samples_s2_batch.tensors.shape)}')
    print(f'    This is a 4D tensor: (T, C, H, W)')
    print(f'    T = seq_len frames stacked together')
    print(f'  .mask shape: {tuple(samples_s2_batch.mask.shape)}')

print(f'\ntargets type: {type(targets_s2_batch)}')
print(f'targets length: {len(targets_s2_batch)}')
print(f'targets[0] type: {type(targets_s2_batch[0])}')
print(f'targets[0] length: {len(targets_s2_batch[0])} (one dict per frame)')

# Show per-frame target shapes
print(f'\nPer-frame targets (frame 0):')
for key, value in targets_s2_batch[0][0].items():
    if isinstance(value, torch.Tensor):
        print(f'  {key}: shape={tuple(value.shape)}')

## 6. Understanding NestedTensor

`NestedTensor` is a wrapper that couples image tensors with their padding mask. This is essential because video frames can have different sizes, and the collate function pads them to a common max size.

In [ ]:
# Demonstrate NestedTensor behavior
print('=== NestedTensor Internals ===')

# Create dummy tensors of different sizes (simulating variable-size images)
img1 = torch.randn(3, 240, 320)  # small
img2 = torch.randn(3, 300, 400)  # larger
img3 = torch.randn(3, 200, 360)  # medium

# NestedTensor.from_tensor_list handles padding
nt = utils.NestedTensor.from_tensor_list([img1, img2, img3])
print(f'Input: 3 images of sizes {tuple(img1.shape)}, {tuple(img2.shape)}, {tuple(img3.shape)}')
print(f'Output .tensors: {tuple(nt.tensors.shape)}')
print(f'Output .mask: {tuple(nt.mask.shape)}')
print(f'\nMask (1=True=padding):')
for i in range(3):
    pad_pixels = nt.mask[i].sum().item()
    total_pixels = nt.mask[i].numel()
    print(f'  Image {i}: {pad_pixels}/{total_pixels} pixels padded ({100*pad_pixels/total_pixels:.1f}%)')

In [ ]:
# Demonstrate select_frame for video clips
print('=== select_frame() for Stage 2 ===')

# Simulate a Stage 2 clip tensor: (T, C, H, W)
clip = torch.randn(8, 3, 300, 400)
mask = torch.zeros(8, 300, 400, dtype=torch.bool)
nt_clip = utils.NestedTensor(clip, mask)

# Extract single frames
frame_0 = nt_clip.select_frame(0)
frame_3 = nt_clip.select_frame(3)

print(f'Full clip: {tuple(nt_clip.tensors.shape)}')
print(f'Frame 0:   {tuple(frame_0.tensors.shape)} (added batch dim: B=1)')
print(f'Frame 3:   {tuple(frame_3.tensors.shape)}')
print(f'Frame 0 mask: {tuple(frame_0.mask.shape)}')

## 7. Complete Tensor Shape Summary

### Stage 1 Pipeline

```
Raw video (decord):         (T_video, H_raw, W_raw, 3) uint8
   |
   v  vr.get_batch([fid, post_fid])
Frame pair:                 (2, H_raw, W_raw, 3) uint8 numpy
   |
   v  transforms (ClipToTensor + Normalize + resize)
Transformed frames:         (3, 2, H', W') float32  [ImageNet normalized]
   |
   v  __getitem__ splits into current + prev
Current image:              (3, H', W') float32
Prev image (in target):     (3, H', W') float32
   |
   v  DataLoader collate_fn -> NestedTensor
Batch images:               (B, 3, H_max, W_max) float32
Batch mask:                 (B, H_max, W_max) bool
Batch targets:              list of B target dicts
  sub_boxes:                (N_i, 4)  cxcywh normalized
  obj_boxes:                (N_i, 4)
  sub_labels:               (N_i,)
  obj_labels:               (N_i,)
  verb_labels:              (N_i, K)  multi-hot
  so_track_ids:             (N_i, 2)
  prev_image:               (3, H', W')  [in each target]
  prev_target:              dict  [in each target]
```

### Stage 2 Pipeline

```
Raw video:                  (T_video, H_raw, W_raw, 3) uint8
   |
   v  vr.get_batch([fid_0, ..., fid_7])
8-frame clip:               (8, H_raw, W_raw, 3) uint8 numpy
   |
   v  transforms
Transformed clip:           (3, 8, H', W') float32
   |
   v  DataLoader collate_fn -> NestedTensor (adds B=1 dim)
Batch clip:                 (8, 3, H', W') float32  [B=1, frames stacked]
Batch mask:                 (8, H', W') bool
Batch targets:              list of 1 element, each = list of 8 target dicts
  Per-frame target:
    sub_boxes:              (N_i, 4)  cxcywh normalized
    unscaled_sub_boxes:     (N_i, 4)  xyxy pixel coords (for ROI Align)
    unscaled_obj_boxes:     (N_i, 4)  xyxy pixel coords
    video_id:               str
    frame_id:               int
    inst_ids:               (N_i,)  relation instance IDs
```

### Key Differences Between Stages

| Aspect | Stage 1 | Stage 2 |
|--------|---------|---------|
| Frames per sample | 2 (pair) | 8 (clip) |
| `samples` format | list of 1 tensor | (T, C, H, W) stacked |
| `targets` format | single dict with `prev_target` | list of 8 dicts (one per frame) |
| `prev_image` in target | Yes (tracking context) | No (GT track IDs used directly) |
| `unscaled_*_boxes` in target | No | Yes (for ROI Align) |
| Batch DataLoader output | samples=(B,3,H,W), targets=list[B dicts] | samples=(B*T,3,H,W), targets=list[1][T dicts] |
| Batch size | 2-4 (multi-GPU) | 1 (single GPU) |

## 8. Visualizing the Bounding Box Formats

Key conversion: annotations start as **xyxy pixel coordinates**, get normalized, then converted to **cxcywh** format by transforms.

In [ ]:
# Demonstrate the box format conversions
print('=== Box Format Conversions ===')

# Raw annotation format (xyxy pixels):
raw_box = torch.tensor([100., 150., 300., 400.])  # x1, y1, x2, y2

# After Normalize transform: converted to cxcywh, normalized by image size
# Suppose image is 640x480:
h, w = 480., 640.
cx = (raw_box[0] + raw_box[2]) / 2 / w  # center x
cy = (raw_box[1] + raw_box[3]) / 2 / h  # center y
bw = (raw_box[2] - raw_box[0]) / w      # width
bh = (raw_box[3] - raw_box[1]) / h      # height
normalized_cxcywh = torch.tensor([cx, cy, bw, bh])

print(f'Raw (xyxy pixels): {raw_box.tolist()}')
print(f'Normalized (cxcywh): {normalized_cxcywh.tolist()}  (all in [0,1])')

# Conversion back (cxcywh -> xyxy)
from util.box_ops import box_cxcywh_to_xyxy
back_to_xyxy = box_cxcywh_to_xyxy(normalized_cxcywh.unsqueeze(0)) * torch.tensor([w, h, w, h])
print(f'Back to xyxy pixels: {back_to_xyxy[0].tolist()}')
print(f'Match: {torch.allclose(back_to_xyxy[0], raw_box, atol=0.1)}')

## 9. Summary

**What we've learned:**

1. **VRDBase** is the main dataset class — `__getitem__` dispatches to `prepare_data_stage1` or `prepare_data_stage2` based on config.
2. **Stage 1 data**: Frame pairs with tracking context. Current frame + previous frame (via `prev_image`/`prev_target`). Returns `(image_tensor, target_dict)` per sample.
3. **Stage 2 data**: 8-frame clips. Returns `(clip_tensor (T,C,H,W), [target_dicts_per_frame])` per sample.
4. **Tracking mechanism**: `so_track_ids` link subject-object pairs across frames. Stage 1 uses these to propagate query embeddings; Stage 2 uses them for temporal pooling in `relation_classifier`.
5. **Box format**: Annotations are xyxy pixels → normalized → cxcywh in [0,1] (DETR standard). Stage 2 keeps unscaled xyxy copies for ROI Align.
6. **NestedTensor**: Wraps tensors + padding masks. `from_tensor_list` pads to common size. `select_frame` extracts frames from clips.
7. **`collate_fn`** from `util/misc.py` batches samples into NestedTensor format.

**Next:** Notebook 3 will build the model and trace tensor shapes through backbone → transformer → prediction heads.